# Session 09: Clustering with K-Means

Practice companion for New. **Synthetic demonstration data**, not the lecturer's original dataset. Run the code cells top to bottom. The original slide snippets remain available separately in the website.

Requires numpy, pandas, matplotlib, seaborn, scikit-learn, and plotly. Install missing packages with `%pip install numpy pandas matplotlib seaborn scikit-learn plotly`.

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)
n = 600
df = pd.DataFrame({
    'order_id': np.arange(1, n+1),
    'customer_id': rng.integers(1, 81, n),
    'date': pd.Timestamp('2025-01-01') + pd.to_timedelta(rng.integers(0, 730, n), unit='D'),
    'product': rng.choice(['Notebook', 'Pen', 'Bag'], n),
    'region': rng.choice(['North', 'South'], n),
    'price': rng.uniform(50, 500, n).round(2),
    'quantity': rng.integers(1, 6, n),
})
df['category'] = df['product'].map({'Notebook': 'Stationery', 'Pen': 'Stationery', 'Bag': 'Accessories'})
df['revenue'] = df['price'] * df['quantity']
df['cost'] = df['revenue'] * rng.uniform(0.45, 0.85, n)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.to_period('M').astype(str)
snapshot = df['date'].max() + pd.Timedelta(days=1)
rfm = df.groupby('customer_id').agg(
    recency=('date', lambda s: (snapshot - s.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('revenue', 'sum'))
# Fixed business thresholds avoid arbitrary tie-breaking; these are illustrative.
rfm['segment'] = np.select(
    [(rfm.recency <= 60) & (rfm.frequency >= 8), rfm.recency > 180],
    ['Champions', 'At risk'], default='Developing')
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
features = rfm[['recency','frequency','monetary']]
X = StandardScaler().fit_transform(features)
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    print(k, round(km.inertia_), round(silhouette_score(X, km.labels_), 3))
# Four is illustrative; choose using diagnostics and business usefulness.
rfm['cluster'] = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(X)
print(rfm.groupby('cluster')[features.columns].mean().round(1))
print(pd.crosstab(rfm.segment, rfm.cluster))
gmm = GaussianMixture(n_components=4, random_state=42).fit(X)
print('Soft memberships:', gmm.predict_proba(X)[:3].round(2))


## Your turn
Scale RFM features, compare candidate K values, fit a final clustering, profile it, and compare it with rule-based segments.

## Watch out
n_init='auto' does not always run several starts; behavior depends on initialization and version. Rules and clusters need not agree. Standardizing does not guarantee meaningful clusters.